## 4. Classification

This section tests **H3**: whether a classifier trained on rhythm and pitch features separates AdP from rock clearly above a majority-class baseline. It then asks which feature family the separation actually comes from.

**Evaluation design.**
- *Grouped cross-validation.* Segments from the same AdP track share tempo, key, instrumentation and production. If segments of one track appeared in both training and test data, a model could recognise the track rather than the style. `StratifiedGroupKFold` with `source_track` as the group keeps every track entirely within one fold, while keeping the class balance in each fold. With 12 AdP tracks and 5 folds, each test fold contains 2–3 AdP tracks.
- *Preprocessing inside the pipeline.* Imputation and standardisation are fitted on the training folds only.
- *No hyperparameter tuning.* Both models use library defaults (C = 1). With only 12 independent AdP sources, tuning would require nested cross-validation and would risk fitting the noise of individual folds.
- *f0-based features are excluded.* They are missing for 44 % of rock clips but only 15 % of AdP clips, so the imputed value would itself reveal the class.

AdP is the positive class, so precision and recall refer to AdP.

### 4.1 Setup

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.pipeline import make_pipeline

# f0-based features exist only where pYIN finds a clear fundamental. They are missing
# for 44 % of rock clips but only 15 % of AdP clips, so any imputed value would itself
# reveal the class. They are used in the analysis (Section 3) but excluded here.
F0_COLS = ["pit_yin_dev_mean", "pit_yin_dev_std", "pit_f0_offgrid_frac",
           "pit_f0_tuning", "pit_f0_median", "pit_f0_iqr"]
MODEL_COLS = [c for c in FEATURE_COLS if c not in F0_COLS]

FEATURE_SETS = {
    "rhythm": [c for c in MODEL_COLS if c.startswith("rhy_")],
    "pitch":  [c for c in MODEL_COLS if c.startswith("pit_")],
    "timbre": [c for c in MODEL_COLS if c.startswith("tim_")],
}
FEATURE_SETS["rhythm + pitch"] = FEATURE_SETS["rhythm"] + FEATURE_SETS["pitch"]  # the H3 feature set
FEATURE_SETS["all"] = MODEL_COLS

y = features["label"].values               # 1 = AdP (positive class), 0 = rock
groups = features["source_track"].values   # keeps all segments of a track in the same fold

# Stratified + grouped: every fold holds out whole tracks AND keeps the class balance.
# With only 12 AdP tracks, each test fold contains 2-3 of them.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)


def make_models():
    """Fresh, unfitted models. Imputation and scaling sit inside the pipeline so they
    are re-fitted on the training folds only and never see the test fold."""
    return {
        "Majority baseline": DummyClassifier(strategy="most_frequent"),
        "Logistic regression": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            LogisticRegression(max_iter=5000)),
        "SVM (RBF)": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            SVC(kernel="rbf", C=1.0, gamma="scale")),
    }


SCORING = {
    "accuracy":  "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall":    make_scorer(recall_score, zero_division=0),
    "f1":        make_scorer(f1_score, zero_division=0),
}


def evaluate(model, cols):
    """Grouped 5-fold CV. Returns mean and std over folds for each metric."""
    res = cross_validate(model, features[cols], y, groups=groups, cv=cv, scoring=SCORING)
    out = {}
    for m in SCORING:
        out[m] = res[f"test_{m}"].mean()
        out[f"{m}_std"] = res[f"test_{m}"].std()
    return out


for name, cols in FEATURE_SETS.items():
    print(f"{name:15s} {len(cols):3d} features")

### 4.2 H3: Rhythm and Pitch Features

In [ ]:
# H3: can a classifier on rhythm and pitch features separate the classes
# clearly above the majority baseline?
h3_cols = FEATURE_SETS["rhythm + pitch"]

rows = []
for name, model in make_models().items():
    r = evaluate(model, h3_cols)
    rows.append({"model": name,
                 **{m: f"{r[m]:.2f} ± {r[m + '_std']:.2f}" for m in SCORING}})

h3_results = pd.DataFrame(rows).set_index("model")
h3_results

### 4.3 Confusion Matrix and Misclassified Clips

The confusion matrix pools the predictions of all five test folds, so every clip is predicted exactly once by a model that never saw its track during training. The list of misclassified clips is the starting point for error analysis: listening to them shows what the model gets wrong.

In [ ]:
# Confusion matrix pooled over the five test folds: every clip is predicted once,
# by a model that never saw its track during training.
h3_model = make_models()["Logistic regression"]
y_pred = cross_val_predict(h3_model, features[h3_cols], y, groups=groups, cv=cv)

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(confusion_matrix(y, y_pred),
                       display_labels=["rock", "adp"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Logistic regression, rhythm + pitch\n(grouped 5-fold CV)")
plt.tight_layout()
plt.savefig(FIGS / "confusion_h3.pdf", bbox_inches="tight")
plt.show()

# Which clips are misclassified? Listening to these is the quickest error analysis.
errors = features.loc[y_pred != y, ["label_name", "source_track", "segment_idx"]]
print(f"{len(errors)} misclassified clips")
errors.sort_values(["label_name", "source_track"])

### 4.4 Ablation by Feature Family

The same models are trained on one feature family at a time. This answers the second part of the research question: which kind of measurable characteristic separates AdP from rock. If timbre alone performs as well as all features together, the classifier relies mainly on sound and production rather than on the rhythmic and tonal properties the hypotheses are about.

In [ ]:
# Ablation: the same models trained on one feature family at a time.
# This answers the second half of the research question - which kind of
# measurable characteristic actually separates AdP from rock.
rows = []
for set_name, cols in FEATURE_SETS.items():
    for model_name, model in make_models().items():
        if model_name == "Majority baseline":
            continue
        r = evaluate(model, cols)
        rows.append({"features": set_name, "model": model_name,
                     "n_features": len(cols), "accuracy": r["accuracy"],
                     "f1": r["f1"], "f1_std": r["f1_std"]})

ablation = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 4.6))
order = list(FEATURE_SETS)
width = 0.38
for i, (model_name, sub) in enumerate(ablation.groupby("model", sort=False)):
    sub = sub.set_index("features").loc[order]
    ax.barh(np.arange(len(order)) + (i - 0.5) * width, sub["f1"], xerr=sub["f1_std"],
            height=width, label=model_name, capsize=3)
ax.set_yticks(np.arange(len(order)), order)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("F1 (AdP as positive class), mean ± std over folds")
ax.set_title("Which feature family separates AdP from rock?")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "ablation.pdf", bbox_inches="tight")
plt.show()

ablation.round(3)

#### Interpretation

**H3 is supported.** A classifier on rhythm and pitch features separates the classes far above the majority baseline: F1 = 0.87 ± 0.02 for logistic regression and 0.90 ± 0.03 for the SVM, against an accuracy of 0.52 for the baseline. Because whole tracks are held out in every fold, the models cannot have succeeded by recognising individual recordings.

**The separation comes mainly from timbre.** In the ablation, rhythm features alone are the weakest family (F1 ≈ 0.71), pitch features reach 0.84, and timbre features alone reach 0.96–0.98. Adding rhythm and pitch to timbre does not improve the result (0.95–0.97 with all features). Rhythm and pitch therefore carry real but secondary information; most of what distinguishes AdP from rock lies in the overall sound of the recordings. The pitch family here consists mainly of chroma statistics, and the measures aimed specifically at microtonality showed no significant difference in Section 3, so the pitch result most likely reflects tonal structure rather than tuning.

**Errors are concentrated.** Of the 25 misclassified clips, 11 are AdP and 14 rock. Five of the eight segments of *Utzp* are misclassified, which suggests that this track is closer to conventional rock than the rest of the material. Three further AdP errors are opening segments (*Ababa hôtel*, *Utzp*, *Sahardnieh*), where an introduction differs from the body of the track. The rock errors are spread over 14 different clips.